# Prediction events, first version

Building the first prediction event dataset: for each inspection with a previous inspection, use only information available before it to predict its outcome. No models here, just the dataset and checks that it doesn't leak the future.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../outputs/inspection_level_longitudinal.csv", parse_dates=["inspectionDate"], low_memory=False)
df = df.sort_values(["estId", "inspectionDate"]).reset_index(drop=True)
print("Loaded rows:", len(df))
print("Establishments:", df["estId"].nunique())
df.head()

Loaded rows: 262547
Establishments: 18898


,estId,oldEstId,estName,address,latitude,longitude,inspectionDate,inspectionStatus,n_infractions,n_minor,n_significant,n_crucial,source
0,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2025-05-22,Pass,0,0,0,0,current
1,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2025-10-02,Pass,2,2,0,0,current
2,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2026-02-03,Pass,1,0,1,0,current
3,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2026-07-14,Conditional Pass,2,0,2,0,current
4,001Vo000013PoSTIA0,10845503.0,KIDULTLAND CAFE,1091 Queen St E None M4M 1K7,43.661798,-79.336098,2026-07-21,Pass,0,0,0,0,current


## Temporarily Not Operating

There is exactly one "Temporarily Not Operating" inspection in the data. We keep it in the establishment's history (it's a real event), but we won't create a prediction row where it is the target, since one example isn't enough to say anything about that outcome.

In [2]:
n_tno = (df["inspectionStatus"] == "Temporarily Not Operating").sum()
print("TNO rows in the full sequence:", n_tno)
df[df["inspectionStatus"] == "Temporarily Not Operating"][["estId", "inspectionDate", "inspectionStatus"]]

TNO rows in the full sequence: 1


,estId,inspectionDate,inspectionStatus
232648,001Vo00001JChH3IAL,2026-02-09,Temporarily Not Operating


## Previous-inspection features

For every inspection, look at the inspection right before it for the same establishment (`shift(1)` within each `estId` group, after sorting by date). The first inspection of each establishment has no previous inspection, so it gets `NaN` here and won't be usable as a prediction row.

In [3]:
grp = df.groupby("estId")

df["prev_inspectionDate"] = grp["inspectionDate"].shift(1)
df["prev_status"] = grp["inspectionStatus"].shift(1)
df["prev_total_infractions"] = grp["n_infractions"].shift(1)
df["prev_minor"] = grp["n_minor"].shift(1)
df["prev_significant"] = grp["n_significant"].shift(1)
df["prev_crucial"] = grp["n_crucial"].shift(1)
df["n_inspections_seen_so_far"] = grp.cumcount()  # number of earlier inspections, 0 for the first one

df["days_since_previous_inspection"] = (df["inspectionDate"] - df["prev_inspectionDate"]).dt.days

# previous outcome as a simple 0/1 flag, easier to use as a feature than the raw label
df["prev_non_pass"] = df["prev_status"].isin(["Conditional Pass", "Closed"]).astype(float)
df.loc[df["prev_status"].isna(), "prev_non_pass"] = np.nan

df[["estId", "inspectionDate", "prev_inspectionDate", "days_since_previous_inspection",
    "prev_status", "prev_non_pass", "prev_total_infractions"]].head(10)

,estId,inspectionDate,prev_inspectionDate,days_since_previous_inspection,prev_status,prev_non_pass,prev_total_infractions
0,001Vo000013PoSTIA0,2025-05-22,NaT,NaN,NaN,NaN,NaN
1,001Vo000013PoSTIA0,2025-10-02,2025-05-22,133.0,Pass,0.0,0.0
2,001Vo000013PoSTIA0,2026-02-03,2025-10-02,124.0,Pass,0.0,2.0
3,001Vo000013PoSTIA0,2026-07-14,2026-02-03,161.0,Pass,0.0,1.0
4,001Vo000013PoSTIA0,2026-07-21,2026-07-14,7.0,Conditional Pass,1.0,2.0
5,001Vo000013PoSUIA0,2025-05-08,NaT,NaN,NaN,NaN,NaN
6,001Vo000013PoSUIA0,2025-11-24,2025-05-08,200.0,Pass,0.0,0.0
7,001Vo000013PoSUIA0,2026-03-09,2025-11-24,105.0,Pass,0.0,0.0
8,001Vo000013PoSVIA0,2025-05-28,NaT,NaN,NaN,NaN,NaN
9,001Vo000013PoSVIA0,2026-06-04,2025-05-28,372.0,Pass,0.0,0.0


## Recent history: previous 365 days

For each target inspection, count inspections, non-pass inspections, total infractions and crucial infractions from the same establishment in the 365 days before the target date. This is done with a running window per establishment: sort by date, then for each row find the earliest previous inspection within 365 days using `np.searchsorted`, and take the cumulative-sum difference between that point and the target row.

Only rows with an index strictly before the target row are ever included, since the establishment's history is sorted by date and dates are unique per establishment.

In [4]:
df["is_non_pass"] = df["inspectionStatus"].isin(["Conditional Pass", "Closed"]).astype(int)

def rolling_365(g):
    g = g.reset_index()
    dates = g["inspectionDate"].values.astype("datetime64[D]")
    total_infr = g["n_infractions"].to_numpy()
    crucial = g["n_crucial"].to_numpy()
    non_pass = g["is_non_pass"].to_numpy()

    cs_infr = np.concatenate([[0], np.cumsum(total_infr)])
    cs_crucial = np.concatenate([[0], np.cumsum(crucial)])
    cs_nonpass = np.concatenate([[0], np.cumsum(non_pass)])

    lower_edge = dates - np.timedelta64(365, "D")
    low_idx = np.searchsorted(dates, lower_edge, side="left")

    idx_arr = np.arange(len(g))
    return pd.DataFrame({
        "orig_index": g["index"],
        "prior_365d_n_inspections": idx_arr - low_idx,
        "prior_365d_n_non_pass": cs_nonpass[idx_arr] - cs_nonpass[low_idx],
        "prior_365d_total_infractions": cs_infr[idx_arr] - cs_infr[low_idx],
        "prior_365d_crucial_infractions": cs_crucial[idx_arr] - cs_crucial[low_idx],
    })

rolled = df.groupby("estId", group_keys=False).apply(rolling_365).set_index("orig_index")
df = df.join(rolled)

df[["estId", "inspectionDate", "prior_365d_n_inspections", "prior_365d_n_non_pass",
    "prior_365d_total_infractions", "prior_365d_crucial_infractions"]].head(10)

,estId,inspectionDate,prior_365d_n_inspections,prior_365d_n_non_pass,prior_365d_total_infractions,prior_365d_crucial_infractions
0,001Vo000013PoSTIA0,2025-05-22,0,0,0,0
1,001Vo000013PoSTIA0,2025-10-02,1,0,0,0
2,001Vo000013PoSTIA0,2026-02-03,2,0,2,0
3,001Vo000013PoSTIA0,2026-07-14,2,0,3,0
4,001Vo000013PoSTIA0,2026-07-21,3,1,5,0
5,001Vo000013PoSUIA0,2025-05-08,0,0,0,0
6,001Vo000013PoSUIA0,2025-11-24,1,0,0,0
7,001Vo000013PoSUIA0,2026-03-09,2,0,0,0
8,001Vo000013PoSVIA0,2025-05-28,0,0,0,0
9,001Vo000013PoSVIA0,2026-06-04,0,0,0,0


## Static features

Keeping this minimal for the first version: just latitude and longitude, already in the inspection-level table. There's no establishment-type or category field that's reliably present across both the current and historical sources, so nothing categorical is added here. No external data (census, OSM, weather, etc) is used.

## Building the target and the prediction rows

A prediction row only exists for inspections that have a previous inspection (`n_inspections_seen_so_far > 0`), and the target inspection itself must not be Temporarily Not Operating.

Target: Pass = 0, Conditional Pass or Closed = 1 (called `target` / non-pass, not "failure").

In [5]:
n_no_prev = (df["n_inspections_seen_so_far"] == 0).sum()

candidates = df[df["n_inspections_seen_so_far"] > 0].copy()
n_before_tno_filter = len(candidates)
pred = candidates[candidates["inspectionStatus"] != "Temporarily Not Operating"].copy()
n_excluded_tno_target = n_before_tno_filter - len(pred)

pred["target"] = pred["inspectionStatus"].isin(["Conditional Pass", "Closed"]).astype(int)

print("Rows with no previous inspection (excluded):", n_no_prev)
print("Rows excluded because the target itself was Temporarily Not Operating:", n_excluded_tno_target)
print("Final prediction rows:", len(pred))

Rows with no previous inspection (excluded): 18898
Rows excluded because the target itself was Temporarily Not Operating: 0
Final prediction rows: 243649


The TNO exclusion count above is 0 because that one TNO inspection happens to be the *first* inspection for its establishment, so it was already dropped by the "no previous inspection" rule. It never had a chance to be a target either way.

## Example establishments

A quick look at a couple of establishments with several inspections, to see the prediction rows next to the raw sequence.

In [6]:
example_est = df["estId"].value_counts().index[0]
print("Example establishment:", example_est)
df[df["estId"] == example_est][["inspectionDate", "inspectionStatus", "n_infractions", "source"]]

Example establishment: 001Vo000013QkY5IAK


,inspectionDate,inspectionStatus,n_infractions,source
113921,2001-02-12,Pass,4,historical
113922,2001-02-14,Pass,0,historical
113923,2001-06-18,Pass,7,historical
113924,2001-06-21,Pass,0,historical
113925,2001-07-11,Pass,1,historical
...,...,...,...,...
114045,2025-09-10,Conditional Pass,3,current
114046,2025-09-18,Pass,2,current
114047,2026-04-21,Conditional Pass,5,current
114048,2026-04-29,Pass,0,current


In [7]:
pred[pred["estId"] == example_est][["inspectionDate", "prev_inspectionDate", "days_since_previous_inspection",
                                       "prev_non_pass", "prior_365d_n_inspections", "target"]]

,inspectionDate,prev_inspectionDate,days_since_previous_inspection,prev_non_pass,prior_365d_n_inspections,target
113922,2001-02-14,2001-02-12,2.0,0.0,1,0
113923,2001-06-18,2001-02-14,124.0,0.0,2,0
113924,2001-06-21,2001-06-18,3.0,0.0,3,0
113925,2001-07-11,2001-06-21,20.0,0.0,4,0
113926,2002-02-01,2001-07-11,205.0,0.0,5,0
...,...,...,...,...,...,...
114045,2025-09-10,2025-07-25,47.0,0.0,3,1
114046,2025-09-18,2025-09-10,8.0,1.0,4,0
114047,2026-04-21,2025-09-18,215.0,0.0,4,1
114048,2026-04-29,2026-04-21,8.0,1.0,5,0


## Leakage and sanity checks

These need to all pass. If any of them fail, something is wrong with the feature construction and it needs fixing before moving on.

In [8]:
# 1. target date is strictly after the previous inspection date
assert (pred["inspectionDate"] > pred["prev_inspectionDate"]).all()

# 2. previous-inspection features are non-negative and come from a real prior row
assert pred["prev_total_infractions"].notna().all()

# 3. 365-day window never has a negative count (would mean the window logic is broken)
assert (pred["prior_365d_n_inspections"] >= 0).all()

# 4. chronological order within each establishment
for _, g in pred.groupby("estId"):
    assert g["inspectionDate"].is_monotonic_increasing

# 5. no duplicate prediction rows for the same establishment + target date
assert pred.duplicated(subset=["estId", "inspectionDate"]).sum() == 0

# 6. Temporarily Not Operating is not a target in the final dataset
assert (pred["inspectionStatus"] != "Temporarily Not Operating").all()

print("All leakage and sanity checks passed.")

All leakage and sanity checks passed.


In [9]:
# brute-force check of the 365-day window against a slow, obviously-correct version,
# on a random sample, to make sure the vectorized version isn't leaking future rows in
rng = np.random.default_rng(0)
sample_idx = rng.choice(pred.index, size=300, replace=False)

mismatches = 0
for i in sample_idx:
    row = pred.loc[i]
    hist = df[(df["estId"] == row["estId"]) & (df["inspectionDate"] < row["inspectionDate"])]
    window = hist[hist["inspectionDate"] >= row["inspectionDate"] - pd.Timedelta(days=365)]
    if len(window) != row["prior_365d_n_inspections"]:
        mismatches += 1
    if window["n_infractions"].sum() != row["prior_365d_total_infractions"]:
        mismatches += 1

print("Brute-force check mismatches on 300 sampled rows (should be 0):", mismatches)

Brute-force check mismatches on 300 sampled rows (should be 0): 0


## Feature columns

These are the columns that would actually go into a model later. Identifiers (`estId`, `oldEstId`) and informative-but-not-a-feature columns (`inspectionDate`, `prev_inspectionDate`, `prev_status`, `source`, raw `inspectionStatus`) are kept in the saved file for traceability but are not in this list.

In [10]:
FEATURE_COLUMNS = [
    "days_since_previous_inspection",
    "prev_non_pass",
    "prev_total_infractions",
    "prev_minor",
    "prev_significant",
    "prev_crucial",
    "n_inspections_seen_so_far",
    "prior_365d_n_inspections",
    "prior_365d_n_non_pass",
    "prior_365d_total_infractions",
    "prior_365d_crucial_infractions",
    "latitude",
    "longitude",
]

identifier_cols = {"estId", "oldEstId", "_id", "unique_id"}
assert identifier_cols.isdisjoint(FEATURE_COLUMNS)
print("Feature columns:", FEATURE_COLUMNS)

Feature columns: ['days_since_previous_inspection', 'prev_non_pass', 'prev_total_infractions', 'prev_minor', 'prev_significant', 'prev_crucial', 'n_inspections_seen_so_far', 'prior_365d_n_inspections', 'prior_365d_n_non_pass', 'prior_365d_total_infractions', 'prior_365d_crucial_infractions', 'latitude', 'longitude']


## Target distribution and feature summary

In [11]:
print(pred["target"].value_counts())
print("Non-pass rate:", round(100 * pred["target"].mean(), 3), "%")

target
0    238969
1      4680
Name: count, dtype: int64
Non-pass rate: 1.921 %


In [12]:
# non-pass rate looks very different between current-sourced and historical-sourced targets,
# worth showing since it affects how the combined dataset should be read later
pred.groupby("source")["target"].agg(["mean", "count"])

,mean,count
source,,
current,0.060531,69766
historical,0.002628,173883


In [13]:
pred[FEATURE_COLUMNS].describe()

,days_since_previous_inspection,prev_non_pass,prev_total_infractions,prev_minor,prev_significant,prev_crucial,n_inspections_seen_so_far,prior_365d_n_inspections,prior_365d_n_non_pass,prior_365d_total_infractions,prior_365d_crucial_infractions,latitude,longitude
count,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000,243649.000000
mean,200.109301,0.020858,0.971443,0.535516,0.338302,0.045327,17.189387,1.826521,0.054061,2.264454,0.147167,43.696685,-79.399098
std,191.623117,0.142909,1.577814,0.892126,0.825828,0.269981,15.752699,1.183740,0.288404,3.754050,0.586414,0.053453,0.089057
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,43.587910,-79.633941
25%,107.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,1.000000,0.000000,0.000000,0.000000,43.653670,-79.449460
50%,152.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12.000000,2.000000,0.000000,1.000000,0.000000,43.679150,-79.396570
75%,227.000000,0.000000,1.000000,1.000000,0.000000,0.000000,25.000000,3.000000,0.000000,3.000000,0.000000,43.738040,-79.348895
max,5632.000000,1.000000,26.000000,10.000000,18.000000,8.000000,128.000000,21.000000,9.000000,87.000000,14.000000,43.839140,-79.130690


## Save the output

In [14]:
output_cols = ["estId", "oldEstId", "inspectionDate", "inspectionStatus", "target",
               "prev_inspectionDate", "prev_status"] + FEATURE_COLUMNS + ["source"]

pred_out = pred[output_cols].sort_values(["estId", "inspectionDate"]).reset_index(drop=True)
pred_out.to_csv("../outputs/prediction_events_v1.csv", index=False)
print("Saved outputs/prediction_events_v1.csv with", len(pred_out), "rows")

Saved outputs/prediction_events_v1.csv with 243649 rows


## Summary

In [15]:
print("Prediction rows:", len(pred_out))
print("Establishments represented:", pred_out["estId"].nunique())
print("Earliest target inspection date:", pred_out["inspectionDate"].min())
print("Latest target inspection date:", pred_out["inspectionDate"].max())
print("Pass targets:", (pred_out["target"] == 0).sum())
print("Non-pass targets:", (pred_out["target"] == 1).sum())
print("Non-pass %:", round(100 * pred_out["target"].mean(), 3))
print("Rows lost, no previous inspection:", n_no_prev)
print("Rows excluded, Temporarily Not Operating target:", n_excluded_tno_target)
print("Feature columns:", FEATURE_COLUMNS)

Prediction rows: 243649
Establishments represented: 16779
Earliest target inspection date: 2001-01-09 00:00:00
Latest target inspection date: 2026-09-08 00:00:00
Pass targets: 238969
Non-pass targets: 4680
Non-pass %: 1.921
Rows lost, no previous inspection: 18898
Rows excluded, Temporarily Not Operating target: 0
Feature columns: ['days_since_previous_inspection', 'prev_non_pass', 'prev_total_infractions', 'prev_minor', 'prev_significant', 'prev_crucial', 'n_inspections_seen_so_far', 'prior_365d_n_inspections', 'prior_365d_n_non_pass', 'prior_365d_total_infractions', 'prior_365d_crucial_infractions', 'latitude', 'longitude']
